In [1]:
!pip install ultralytics opencv-python-headless streamlit pyyaml tqdm

   ---------------------------------------- 0.0/39.4 MB ? eta -:--:--
   ---------------------------------------- 0.3/39.4 MB ? eta -:--:--
    --------------------------------------- 0.5/39.4 MB 1.7 MB/s eta 0:00:24
    --------------------------------------- 0.8/39.4 MB 1.5 MB/s eta 0:00:26
   - -------------------------------------- 1.0/39.4 MB 1.5 MB/s eta 0:00:26
   - -------------------------------------- 1.3/39.4 MB 1.3 MB/s eta 0:00:29
   - -------------------------------------- 1.3/39.4 MB 1.3 MB/s eta 0:00:29
   - -------------------------------------- 1.3/39.4 MB 1.3 MB/s eta 0:00:29
   - -------------------------------------- 1.6/39.4 MB 911.5 kB/s eta 0:00:42
   - -------------------------------------- 1.6/39.4 MB 911.5 kB/s eta 0:00:42
   - -------------------------------------- 1.8/39.4 MB 890.6 kB/s eta 0:00:43
   -- ------------------------------------- 2.1/39.4 MB 896.4 kB/s eta 0:00:42
   -- ------------------------------------- 2.4/39.4 MB 900.8 kB/s eta 0:00:42
   

In [ ]:
import os
import shutil
import random
from pathlib import Path
from tqdm import tqdm

# Paths (Change these to match your dataset location)
DATASET_PATH = r"C:\Users\rashi\Desktop\LPU\Capstone\animal__poacher_dataset"  # Source dataset path
YOLOV8_FORMAT_PATH = r"C:\Users\rashi\Desktop\LPU\Capstone\yolov8-dataset"  # Destination for YOLOv8 format

# Ensure output directories exist
for split in ["train", "val", "test"]:
    os.makedirs(f"{YOLOV8_FORMAT_PATH}/{split}/images", exist_ok=True)
    os.makedirs(f"{YOLOV8_FORMAT_PATH}/{split}/labels", exist_ok=True)

# Train-Test-Val split
for split in ["train", "test"]:  # No 'val' in source dataset, we'll create it from 'train'
    split_path = os.path.join(DATASET_PATH, split)
    
    if not os.path.exists(split_path):
        print(f"⚠️ Warning: '{split}' directory not found, skipping...")
        continue

    print(f"📌 Processing '{split}' split...")

    for label in tqdm(os.listdir(split_path), desc=f'Processing {split}'):
        label_path = os.path.join(split_path, label)
        images = [f for f in os.listdir(label_path) if f.endswith(('.jpg', '.jpeg', '.png'))]
        annotations_path = os.path.join(label_path, "Label")

        if not os.path.exists(annotations_path):
            print(f"⚠️ Warning: No label directory found for '{label}' in '{split}', skipping...")
            continue

        # Shuffle images to ensure randomness
        random.shuffle(images)
        val_split_idx = int(0.8 * len(images))  # 80% Train, 20% Val

        for i, img in enumerate(images):
            img_name = Path(img).stem
            img_src = os.path.join(label_path, img)

            # Move labels
            label_file = f"{img_name}.txt"
            label_src = os.path.join(annotations_path, label_file)

            # Assign images and labels to train or val
            if split == "train" and i >= val_split_idx:
                target_split = "val"
            else:
                target_split = split

            img_dest = os.path.join(YOLOV8_FORMAT_PATH, target_split, "images", img)
            label_dest = os.path.join(YOLOV8_FORMAT_PATH, target_split, "labels", label_file)

            shutil.copy(img_src, img_dest)  # Copy image
            if os.path.exists(label_src):
                shutil.copy(label_src, label_dest)  # Copy label
            else:
                print(f"⚠️ Warning: No annotation found for '{img_name}' in '{split}', skipping label...")

print("\n✅ Dataset conversion completed successfully!")


📌 Processing 'train' split...


Processing train: 100%|██████████| 27/27 [01:48<00:00,  4.01s/it]


📌 Processing 'test' split...


Processing test: 100%|██████████| 27/27 [00:36<00:00,  1.37s/it]


✅ Dataset conversion completed successfully!


In [1]:
import os
import glob
from PIL import Image

# Path to your image dataset
IMAGE_DIR = "C:/Users/rashi/Desktop/LPU/Capstone/yolov8-dataset/test/images"  # Update this

def check_image_sizes():
    """ Check the dimensions of all images in the dataset """
    image_files = glob.glob(os.path.join(IMAGE_DIR, "*"))
    
    sizes = set()
    
    for img_file in image_files:
        with Image.open(img_file) as img:
            sizes.add(img.size)  # (width, height)
    
    if len(sizes) == 1:
        print(f"✅ All images have the same size: {sizes.pop()}")
    else:
        print(f"⚠️ Images have different sizes: {sizes}")

if __name__ == "__main__":
    check_image_sizes()


KeyboardInterrupt: 

In [13]:
import os
import glob
from PIL import Image

# Paths to dataset
IMAGE_DIR = "C:/Users/rashi/Desktop/LPU/Capstone/yolov8-dataset/val/images"  # Update this
LABEL_DIR = "C:/Users/rashi/Desktop/LPU/Capstone/yolov8-dataset/val/labels"  # Update this
OUTPUT_DIR = "C:/Users/rashi/Desktop/LPU/Capstone/yolov8-dataset/val/labels_normalized"  # New directory for normalized labels

os.makedirs(OUTPUT_DIR, exist_ok=True)  # Create output folder if not exists

def normalize_bboxes():
    """ Fetch image sizes and normalize bounding boxes """
    for img_file in glob.glob(os.path.join(IMAGE_DIR, "*.jpg")):  # Change extension if needed
        img_name = os.path.basename(img_file).replace(".jpg", ".txt")  # Match label file
        label_file = os.path.join(LABEL_DIR, img_name)
        
        if not os.path.exists(label_file):
            print(f"⚠️ No label file for {img_name}, skipping...")
            continue
        
        # Get image width & height
        with Image.open(img_file) as img:
            W, H = img.size  # (width, height)
        
        normalized_lines = []
        delete_flag = False  # Track if we need to delete the file

        try:
            with open(label_file, "r") as f:
                for line in f:
                    values = line.strip().split()
                    
                    # If label doesn't have exactly 5 values (class_id + 4 coordinates), mark it for deletion
                    if len(values) != 5:
                        print(f"❌ Incorrect format in {label_file}: {values}")
                        delete_flag = True
                        break
                    
                    class_id = values[0]  # Class number
                    
                    # Try to convert coordinates to float, if error, mark for deletion
                    try:
                        x_min, y_min, x_max, y_max = map(float, values[1:])
                    except ValueError:
                        print(f"❌ Invalid bounding box values in {label_file}: {values}")
                        delete_flag = True
                        break
                    
                    # Normalize
                    x_center = (x_min + x_max) / (2 * W)
                    y_center = (y_min + y_max) / (2 * H)
                    bbox_width = (x_max - x_min) / W
                    bbox_height = (y_max - y_min) / H
                    
                    normalized_lines.append(f"{class_id} {x_center:.6f} {y_center:.6f} {bbox_width:.6f} {bbox_height:.6f}")

            # If any issue was found, delete the label & image
            if delete_flag:
                raise ValueError("Invalid label format")

            # Save normalized labels
            with open(os.path.join(OUTPUT_DIR, img_name), "w") as f_out:
                f_out.write("\n".join(normalized_lines))

            print(f"✅ Normalized {img_name}")

        except Exception as e:
            print(f"❌ Deleting {label_file} and corresponding image due to error: {e}")
            os.remove(label_file)  # Delete the label file
            os.remove(img_file)  # Delete the corresponding image

if __name__ == "__main__":
    normalize_bboxes()

✅ Normalized 000107d1d5876a7d.txt
✅ Normalized 0040d7f1676ab75c.txt
✅ Normalized 004793df03b93a02.txt
✅ Normalized 0056496ee24a4912.txt
✅ Normalized 007856a72e863b13.txt
✅ Normalized 00994395b5833be1.txt
✅ Normalized 00cfc4ac9b70768c.txt
✅ Normalized 00d5a97ac2d2d837.txt
✅ Normalized 00e5c9e716916b2b.txt
✅ Normalized 00f6541338e44d7d.txt
✅ Normalized 00fb001de33a7b71.txt
✅ Normalized 010739d688d6ff46.txt
✅ Normalized 011b496d23d11acf.txt
✅ Normalized 011e915374f14b7d.txt
✅ Normalized 012ce713bdee60d0.txt
✅ Normalized 013240b8dc5ed267.txt
✅ Normalized 0136030c6137f6df.txt
❌ Incorrect format in C:/Users/rashi/Desktop/LPU/Capstone/yolov8-dataset/val/labels\01399dab5c3366e4.txt: ['18', 'bear', '312.32', '235.02983000000003', '689.28', '664.102825']
❌ Deleting C:/Users/rashi/Desktop/LPU/Capstone/yolov8-dataset/val/labels\01399dab5c3366e4.txt and corresponding image due to error: Invalid label format
✅ Normalized 01405e94ad939f04.txt
✅ Normalized 016a832c6088b543.txt
✅ Normalized 01b7d115565

In [1]:
from ultralytics import YOLO

# Load YOLOv8n model (nano version)
model = YOLO("yolov8n.pt")  
config_path=r"C:\Users\rashi\Desktop\LPU\Capstone\yolov8-dataset\data.yaml"

# Train the model
results = model.train(data=config_path, epochs=50, imgsz=640)

c:\Users\rashi\AppData\Local\Programs\Python\Python39\lib\site-packages\torch\utils\_pytree.py:185: FutureWarning: optree is installed but the version is too old to support PyTorch Dynamo in C++ pytree. C++ pytree support is disabled. Please consider upgrading optree using `python3 -m pip install --upgrade 'optree>=0.13.0'`.
  warnings.warn(


New https://pypi.org/project/ultralytics/8.3.83 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.74  Python-3.9.13 torch-2.6.0+cpu CPU (11th Gen Intel Core(TM) i5-1135G7 2.40GHz)
engine\trainer: task=detect, mode=train, model=yolov8n.pt, data=C:\Users\rashi\Desktop\LPU\Capstone\yolov8-dataset\data.yaml, epochs=50, time=None, patience=100, batch=16, imgsz=640, save=True, save_period=-1, cache=False, device=None, workers=8, project=None, name=train25, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retin

train: Scanning C:\Users\rashi\Desktop\LPU\Capstone\yolov8-dataset\train\labels.cache... 3881 images, 31 backgrounds, 0 corrupt: 100%|██████████| 3912/3912 [00:00<?, ?it/s]
val: Scanning C:\Users\rashi\Desktop\LPU\Capstone\yolov8-dataset\val\labels.cache... 1002 images, 8 backgrounds, 0 corrupt: 100%|██████████| 1010/1010 [00:00<?, ?it/s]

val: WARNING  C:\Users\rashi\Desktop\LPU\Capstone\yolov8-dataset\val\images\326b9a3496142b2e.jpg: 1 duplicate labels removed


Plotting labels to runs\detect\train25\labels.jpg... 
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.000323, momentum=0.9) with parameter groups 57 weight(decay=0.0), 64 weight(decay=0.0005), 63 bias(decay=0.0)
TensorBoard: model graph visualization added 
Image sizes 640 train, 640 val
Using 0 dataloader workers
Logging results to runs\detect\train25
Starting training for 50 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


...

                   all       1010       1096      0.762      0.748      0.805      0.697



50 epochs completed in 18.767 hours.
Optimizer stripped from runs\detect\train25\weights\last.pt, 6.2MB
Optimizer stripped from runs\detect\train25\weights\best.pt, 6.2MB

Validating runs\detect\train25\weights\best.pt...
Ultralytics 8.3.74  Python-3.9.13 torch-2.6.0+cpu CPU (11th Gen Intel Core(TM) i5-1135G7 2.40GHz)
Model summary (fused): 168 layers, 3,010,913 parameters, 0 gradients, 8.1 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 32/32 [01:11<00:00,  2.24s/it]


                   all       1010       1096      0.788      0.734      0.809      0.701
                  Bear         18         19      0.737      0.684      0.649      0.631
                  Bull         10         10      0.857        0.7       0.75      0.663
                 Camel         14         15      0.612        0.6      0.572      0.543
               Cheetah         25         25      0.674      0.746      0.743      0.624
                  Deer         65         88      0.741      0.818       0.82      0.681
                   Fox         30         31       0.83       0.79      0.913      0.822
               Giraffe         60         70      0.971      0.941      0.964      0.792
          Hippopotamus         16         19      0.849      0.737      0.845      0.721
                Jaguar         18         18      0.291      0.274       0.42      0.387
              Kangaroo         20         22      0.528      0.591      0.603      0.504
                 Koal

SyntaxError: '[31m[1msave_path[0m' is not a valid YOLO argument. Similar arguments are i.e. ['save_txt=False', 'save_crop=False', 'save=True'].

    Arguments received: ['yolo', '--f=c:\\Users\\rashi\\AppData\\Roaming\\jupyter\\runtime\\kernel-v3f13ce959b13567a1cd93bea66a20dc2052e1c549.json']. Ultralytics 'yolo' commands use the following syntax:

        yolo TASK MODE ARGS

        Where   TASK (optional) is one of frozenset({'classify', 'obb', 'detect', 'segment', 'pose'})
                MODE (required) is one of frozenset({'val', 'track', 'benchmark', 'train', 'predict', 'export'})
                ARGS (optional) are any number of custom 'arg=value' pairs like 'imgsz=320' that override defaults.
                    See all ARGS at https://docs.ultralytics.com/usage/cfg or with 'yolo cfg'

    1. Train a detection model for 10 epochs with an initial learning_rate of 0.01
        yolo train data=coco8.yaml model=yolo11n.pt epochs=10 lr0=0.01

    2. Predict a YouTube video using 

In [2]:
metrics = model.val()
print(metrics)

Ultralytics 8.3.74  Python-3.9.13 torch-2.6.0+cpu CPU (11th Gen Intel Core(TM) i5-1135G7 2.40GHz)
Model summary (fused): 168 layers, 3,010,913 parameters, 0 gradients, 8.1 GFLOPs


val: Scanning C:\Users\rashi\Desktop\LPU\Capstone\yolov8-dataset\val\labels.cache... 1002 images, 8 backgrounds, 0 corrupt: 100%|██████████| 1010/1010 [00:00<?, ?it/s]

val: WARNING  C:\Users\rashi\Desktop\LPU\Capstone\yolov8-dataset\val\images\326b9a3496142b2e.jpg: 1 duplicate labels removed



                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 64/64 [01:05<00:00,  1.03s/it]


                   all       1010       1096      0.787      0.737       0.81      0.701
                  Bear         18         19      0.747      0.737      0.697      0.677
                  Bull         10         10      0.855        0.7      0.752      0.664
                 Camel         14         15      0.605        0.6      0.572      0.543
               Cheetah         25         25      0.675      0.747      0.743      0.624
                  Deer         65         88       0.74      0.818      0.818      0.682
                   Fox         30         31      0.826      0.764      0.911      0.818
               Giraffe         60         70      0.971      0.942      0.964      0.794
          Hippopotamus         16         19      0.793      0.737      0.837      0.711
                Jaguar         18         18      0.295      0.279       0.42      0.387
              Kangaroo         20         22        0.5      0.591       0.59      0.493
                 Koal

In [21]:
import shutil
import os

# Specify the source and destination paths
source_path = r"runs/detect/train25/weights/best.pt"  # Path to the best.pt in the training directory
destination_path = r"C:/Users/rashi/Desktop/LPU/Capstone/best.pt"  # Desired location

# Create destination directory if it doesn't exist
os.makedirs(os.path.dirname(destination_path), exist_ok=True)

# Copy the best model weights to the new location
shutil.copy(source_path, destination_path)

# Print confirmation
print(f"Best model weights saved to: {destination_path}")


Best model weights saved to: C:/Users/rashi/Desktop/LPU/Capstone/best.pt


In [2]:
from ultralytics import YOLO

# Load YOLOv8n model (nano version)
model = YOLO("yolov8n.pt")  
config_path=r"C:\Users\rashi\Desktop\LPU\Capstone\yolov8-dataset\data.yaml"

# Train the model
results = model.train(data=config_path, epochs=70, imgsz=640)


KeyboardInterrupt

